# MATH 6243 — Project EDA
## Predicting Personal Film Preferences from Letterboxd Ratings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('enriched_ratings.csv')

for col in ['imdb_rating', 'rt_score', 'tmdb_vote_avg', 'runtime', 'budget', 'tmdb_popularity']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Multi-class label
def bucket(r):
    if r <= 2.0:   return '1 - Disliked'
    elif r <= 3.0: return '2 - Mixed'
    elif r <= 4.0: return '3 - Liked'
    else:          return '4 - Loved'

df['label'] = df['rating'].apply(bucket)
label_order = ['1 - Disliked', '2 - Mixed', '3 - Liked', '4 - Loved']

print(df.shape)
df.head()

## 1. Rating Distribution & Multi-class Bucketing

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Raw half-star distribution
counts = df['rating'].value_counts().sort_index()
axes[0].bar(counts.index, counts.values, width=0.35,
            color=sns.color_palette('muted')[0], edgecolor='white')
axes[0].set_xlabel('Star Rating')
axes[0].set_ylabel('Count')
axes[0].set_title('Raw Rating Distribution (0.5–5)')
axes[0].set_xticks(sorted(df['rating'].unique()))

# 4-class bucketed distribution
label_counts = df['label'].value_counts().reindex(label_order)
palette = sns.color_palette('muted', 4)
bars = axes[1].bar(label_counts.index, label_counts.values, color=palette, edgecolor='white')
for bar, val in zip(bars, label_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val}\n({val/len(df)*100:.0f}%)', ha='center', va='bottom', fontsize=9)
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
axes[1].set_title('Multi-class Label Distribution')
axes[1].tick_params(axis='x', labelsize=9)

plt.tight_layout()
plt.savefig('fig1_rating_distribution.png', bbox_inches='tight')
plt.show()

print(label_counts)
print(f'\nBinary split (>=3.5): {df["liked"].sum()} liked, {(~df["liked"].astype(bool)).sum()} not liked')

## 2. Missing Data Audit

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]

fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(missing.index, missing.values, color=sns.color_palette('muted')[3])
for i, v in enumerate(missing.values):
    ax.text(v + 0.1, i, f'{v} ({v/len(df)*100:.0f}%)', va='center', fontsize=9)
ax.set_xlabel('Missing count')
ax.set_title('Missing Values by Feature')
ax.set_xlim(0, missing.max() * 1.5)
plt.tight_layout()
plt.savefig('fig2_missing_values.png', bbox_inches='tight')
plt.show()

# Note: missing RT scores are likely OMDB mapping failures, not absent RT pages.
print('Films missing RT score:')
print(df[df['rt_score'].isnull()][['letterboxd_title', 'letterboxd_year', 'rating']].to_string(index=False))

## 3. Critical Scores vs. Personal Rating

In [ ]:
score_features = [
    ('imdb_rating',   'IMDB Rating'),
    ('rt_score',      'RT Score'),
    ('tmdb_vote_avg', 'TMDB Avg Rating'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (feat, label) in zip(axes, score_features):
    valid = df[['rating', feat, 'label']].dropna()
    ax.scatter(
        valid[feat], valid['rating'],
        c=valid['label'].map({'1 - Disliked': 0, '2 - Mixed': 1, '3 - Liked': 2, '4 - Loved': 3}),
        cmap='RdYlGn', alpha=0.7, edgecolors='none', s=40
    )
    m, b = np.polyfit(valid[feat], valid['rating'], 1)
    xs = np.linspace(valid[feat].min(), valid[feat].max(), 100)
    ax.plot(xs, m*xs + b, color='steelblue', linewidth=1.5, linestyle='--')
    corr = valid['rating'].corr(valid[feat])
    ax.set_xlabel(label)
    ax.set_ylabel('Personal Rating')
    ax.set_title(f'{label}  (r = {corr:.2f})')

plt.suptitle('Critical Consensus Scores vs. Personal Rating', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig3_scores_vs_rating.png', bbox_inches='tight')
plt.show()

## 4. Genre Analysis

In [ ]:
genre_df = df[['letterboxd_title', 'rating', 'label', 'genres']].dropna(subset=['genres'])
genre_df = genre_df.assign(genre=genre_df['genres'].str.split('|')).explode('genre')
genre_df['genre'] = genre_df['genre'].str.strip()

genre_counts = genre_df['genre'].value_counts()
genre_stats  = genre_df.groupby('genre')['rating'].agg(['mean', 'count'])
genre_stats  = genre_stats[genre_stats['count'] >= 3].sort_values('mean', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Frequency
top = genre_counts.head(12)
axes[0].barh(top.index[::-1], top.values[::-1], color=sns.color_palette('muted')[0])
axes[0].set_xlabel('Number of Films')
axes[0].set_title('Top Genres by Frequency')

# Mean rating
overall_mean = df['rating'].mean()
bar_colors = ['seagreen' if m >= overall_mean else 'tomato' for m in genre_stats['mean']]
axes[1].barh(genre_stats.index[::-1], genre_stats['mean'][::-1], color=bar_colors[::-1])
axes[1].axvline(overall_mean, color='steelblue', linestyle='--', linewidth=1.2,
                label=f'Overall mean ({overall_mean:.2f})')
for i, (_, row) in enumerate(genre_stats[::-1].iterrows()):
    axes[1].text(row['mean'] + 0.02, i, f"n={int(row['count'])}", va='center', fontsize=7)
axes[1].set_xlabel('Mean Personal Rating')
axes[1].set_title('Mean Rating by Genre (min. 3 films)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('fig4_genre_analysis.png', bbox_inches='tight')
plt.show()

## 5. Language & Country of Origin

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# --- Language frequency ---
lang_counts = df['original_language'].value_counts().head(10)
axes[0,0].bar(lang_counts.index, lang_counts.values,
              color=sns.color_palette('muted')[1], edgecolor='white')
axes[0,0].set_title('Films by Original Language')
axes[0,0].set_ylabel('Count')

# --- Language mean rating ---
lang_means = df.groupby('original_language')['rating'].mean().reindex(lang_counts.index)
bar_colors = ['seagreen' if m >= overall_mean else 'tomato' for m in lang_means]
axes[0,1].bar(lang_means.index, lang_means.values, color=bar_colors, edgecolor='white')
axes[0,1].axhline(overall_mean, color='steelblue', linestyle='--', linewidth=1.2,
                  label=f'Overall mean ({overall_mean:.2f})')
axes[0,1].set_title('Mean Rating by Language')
axes[0,1].set_ylabel('Mean Personal Rating')
axes[0,1].legend(fontsize=8)

# --- Country frequency ---
country_counts = df['country'].value_counts().head(10)
axes[1,0].bar(country_counts.index, country_counts.values,
              color=sns.color_palette('muted')[2], edgecolor='white')
axes[1,0].set_title('Films by Country of Origin')
axes[1,0].set_ylabel('Count')
axes[1,0].tick_params(axis='x', rotation=25)

# --- Country mean rating ---
country_means = df.groupby('country')['rating'].mean().reindex(country_counts.index)
bar_colors2 = ['seagreen' if m >= overall_mean else 'tomato' for m in country_means]
axes[1,1].bar(country_means.index, country_means.values, color=bar_colors2, edgecolor='white')
axes[1,1].axhline(overall_mean, color='steelblue', linestyle='--', linewidth=1.2,
                  label=f'Overall mean ({overall_mean:.2f})')
axes[1,1].set_title('Mean Rating by Country')
axes[1,1].set_ylabel('Mean Personal Rating')
axes[1,1].tick_params(axis='x', rotation=25)
axes[1,1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('fig5_language_country.png', bbox_inches='tight')
plt.show()

## 6. Director Analysis

In [ ]:
dir_stats = df.groupby('director')['rating'].agg(['mean', 'count']).sort_values('count', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Directors with most films watched
top_watched = dir_stats[dir_stats['count'] >= 2].head(15)
axes[0].barh(top_watched.index[::-1], top_watched['count'][::-1],
             color=sns.color_palette('muted')[4])
axes[0].set_xlabel('Films Watched')
axes[0].set_title('Most Watched Directors (min. 2 films)')

# Mean rating for directors with >= 2 films, sorted by rating
top_rated = dir_stats[dir_stats['count'] >= 2].sort_values('mean', ascending=False).head(15)
bar_colors = ['seagreen' if m >= overall_mean else 'tomato' for m in top_rated['mean']]
axes[1].barh(top_rated.index[::-1], top_rated['mean'][::-1], color=bar_colors[::-1])
axes[1].axvline(overall_mean, color='steelblue', linestyle='--', linewidth=1.2,
                label=f'Overall mean ({overall_mean:.2f})')
for i, (_, row) in enumerate(top_rated[::-1].iterrows()):
    axes[1].text(row['mean'] + 0.02, i, f"n={int(row['count'])}", va='center', fontsize=7)
axes[1].set_xlabel('Mean Personal Rating')
axes[1].set_title('Mean Rating by Director (min. 2 films)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('fig6_director_analysis.png', bbox_inches='tight')
plt.show()

print('All directors with 2+ films watched:')
print(dir_stats[dir_stats['count'] >= 2].sort_values('mean', ascending=False)
      .rename(columns={'mean': 'avg_rating', 'count': 'films_watched'}).to_string())

## 7. Consensus Divergence Tables

In [ ]:
# Normalise personal rating to 0-10 for comparison with IMDB
df['rating_norm'] = df['rating'] * 2

cols = ['letterboxd_title', 'letterboxd_year', 'rating', 'rating_norm']

# --- IMDB divergence ---
imdb = df[cols + ['imdb_rating']].dropna(subset=['imdb_rating']).copy()
imdb['divergence'] = (imdb['rating_norm'] - imdb['imdb_rating']).round(2)

print('=== Rated MUCH HIGHER than IMDB ===')
print(imdb.nlargest(8, 'divergence')[['letterboxd_title', 'letterboxd_year', 'rating', 'imdb_rating', 'divergence']].to_string(index=False))
print()
print('=== Rated MUCH LOWER than IMDB ===')
print(imdb.nsmallest(15, 'divergence')[['letterboxd_title', 'letterboxd_year', 'rating', 'imdb_rating', 'divergence']].to_string(index=False))

In [ ]:
# --- RT divergence (RT is 0-100, personal is 0-10, scale RT to 0-10) ---
rt = df[cols + ['rt_score']].dropna(subset=['rt_score']).copy()
rt['rt_score_norm'] = rt['rt_score'] / 10
rt['divergence'] = (rt['rating_norm'] - rt['rt_score_norm']).round(2)

print('=== Rated MUCH HIGHER than RT ===')
print(rt.nlargest(8, 'divergence')[['letterboxd_title', 'letterboxd_year', 'rating', 'rt_score', 'divergence']].to_string(index=False))
print()
print('=== Rated MUCH LOWER than RT ===')
print(rt.nsmallest(8, 'divergence')[['letterboxd_title', 'letterboxd_year', 'rating', 'rt_score', 'divergence']].to_string(index=False))

## 8. Correlation Heatmap

In [ ]:
num_cols = ['rating', 'imdb_rating', 'rt_score', 'tmdb_vote_avg',
            'runtime', 'release_year', 'tmdb_popularity', 'budget']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Correlation Matrix — Numeric Features', fontweight='bold')
plt.tight_layout()
plt.savefig('fig7_correlation_heatmap.png', bbox_inches='tight')
plt.show()

## 9. EDA Summary

In [ ]:
print('=== Quick Summary ===')
print(f'Total films:        {len(df)}')
print(f'Rating range:       {df["rating"].min()} – {df["rating"].max()}')
print(f'Mean rating:        {df["rating"].mean():.2f}')
print(f'Unique directors:   {df["director"].nunique()}')
print(f'Unique languages:   {df["original_language"].nunique()}')
print(f'Unique countries:   {df["country"].nunique()}')
print(f'\nMulti-class distribution:')
print(df['label'].value_counts().reindex(label_order))
print(f'\nCorrelation with personal rating:')
for col, name in [('imdb_rating','IMDB'), ('rt_score','RT'), ('tmdb_vote_avg','TMDB Avg')]:
    print(f'  {name}: {df["rating"].corr(df[col]):.3f}')

## 10. Watchlist Scoring

Apply the trained models to `enriched_watchlist.csv` to produce a ranked list of films most likely to be enjoyed.

**Approach:** Refit the best binary model (Random Forest, AUC = 0.711) on the full ratings dataset, then apply the same feature engineering pipeline to the watchlist and score each film with `predict_proba`.

In [ ]:
# Load watchlist
wl = pd.read_csv('enriched_watchlist.csv')

for col in ['imdb_rating', 'rt_score', 'tmdb_vote_avg', 'runtime',
            'release_year', 'tmdb_popularity', 'budget', 'awards']:
    wl[col] = pd.to_numeric(wl[col], errors='coerce')

print(f'{len(wl)} films in watchlist')
missing = wl[['imdb_rating', 'rt_score', 'tmdb_vote_avg', 'runtime', 'release_year', 'awards']].isnull().sum()
print(f'Missing values:\n{missing[missing > 0]}')

In [ ]:
# ── Feature engineering (self-contained, mirrors training pipeline) ───────────

from sklearn.preprocessing import MultiLabelBinarizer

# Re-fit MLB on training data genres
df_train = pd.read_csv('enriched_ratings.csv')
train_genre_lists = df_train['genres'].fillna('').apply(
    lambda x: [g.strip() for g in x.split('|') if g.strip()]
)
mlb_wl = MultiLabelBinarizer()
mlb_wl.fit(train_genre_lists)

# Genre OHE on training to get the >=5 column filter
train_genre_ohe = pd.DataFrame(mlb_wl.transform(train_genre_lists),
                                columns=[f'genre_{g}' for g in mlb_wl.classes_])
keep_genres = train_genre_ohe.columns[train_genre_ohe.sum() >= 5].tolist()

# Genre OHE on watchlist
wl_genre_lists = wl['genres'].fillna('').apply(
    lambda x: [g.strip() for g in x.split('|') if g.strip()]
)
wl_genre_ohe = pd.DataFrame(mlb_wl.transform(wl_genre_lists),
                              columns=[f'genre_{g}' for g in mlb_wl.classes_],
                              index=wl.index)[keep_genres]

# Language OHE
top_langs_wl = df_train['original_language'].value_counts().head(5).index.tolist()
wl['language_collapsed'] = wl['original_language'].apply(
    lambda x: x if x in top_langs_wl else 'other'
)
train_lang_ohe = pd.get_dummies(
    df_train['original_language'].apply(lambda x: x if x in top_langs_wl else 'other'),
    prefix='lang'
)
wl_lang_ohe = pd.get_dummies(wl['language_collapsed'], prefix='lang')
for col in train_lang_ohe.columns:
    if col not in wl_lang_ohe.columns:
        wl_lang_ohe[col] = 0
wl_lang_ohe = wl_lang_ohe[train_lang_ohe.columns]

# Numeric features
numeric_cols_wl = ['imdb_rating', 'rt_score', 'tmdb_vote_avg', 'runtime', 'release_year', 'awards']
X_wl_base = pd.concat([wl[numeric_cols_wl], wl_genre_ohe, wl_lang_ohe], axis=1).copy()

print(f'Watchlist feature matrix: {X_wl_base.shape}')

In [ ]:
# ── Refit full pipeline on all 166 rated films (self-contained) ───────────────

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

SEED = 42

# Reload training data
df = pd.read_csv('enriched_ratings.csv')
for col in ['imdb_rating', 'rt_score', 'tmdb_vote_avg', 'runtime',
            'release_year', 'tmdb_popularity', 'budget', 'awards']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

y_binary = (df['rating'] >= 3.5).astype(int)

# Rebuild training feature matrix using same genre/language setup from cell above
train_genre_lists = df['genres'].fillna('').apply(
    lambda x: [g.strip() for g in x.split('|') if g.strip()]
)
train_genre_ohe_full = pd.DataFrame(mlb_wl.transform(train_genre_lists),
                                     columns=[f'genre_{g}' for g in mlb_wl.classes_])[keep_genres]

df['language_collapsed'] = df['original_language'].apply(
    lambda x: x if x in top_langs_wl else 'other'
)
train_lang_ohe_full = pd.get_dummies(df['language_collapsed'], prefix='lang')
for col in wl_lang_ohe.columns:
    if col not in train_lang_ohe_full.columns:
        train_lang_ohe_full[col] = 0
train_lang_ohe_full = train_lang_ohe_full[wl_lang_ohe.columns]

X_base = pd.concat([df[numeric_cols_wl], train_genre_ohe_full, train_lang_ohe_full], axis=1).copy()

# Impute
imp_train = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(imp_train.fit_transform(X_base), columns=X_base.columns)

# Director target encoding
dir_means_train = df.groupby('director')['rating'].mean()
global_mean_train = df['rating'].mean()
dir_enc_train = df['director'].map(dir_means_train).fillna(global_mean_train).values.reshape(-1, 1)
X_train_full = np.hstack([X_train_imp.values, dir_enc_train])

# Scale
scaler_final = StandardScaler()
X_train_scaled = scaler_final.fit_transform(X_train_full)

# Fit
rf_final = RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=3,
                                   random_state=SEED, n_jobs=-1)
rf_final.fit(X_train_scaled, y_binary)
print('Model refit on full training data.')

In [ ]:
# ── Apply same pipeline to watchlist ─────────────────────────────────────────

# Impute watchlist using training imputer
X_wl_imp = pd.DataFrame(
    imp_train.transform(X_wl_base),
    columns=X_wl_base.columns
)

# Director target encoding using training director means
dir_enc_wl = wl['director'].map(dir_means_train).fillna(global_mean_train).values.reshape(-1, 1)
X_wl_full = np.hstack([X_wl_imp.values, dir_enc_wl])

# Scale using training scaler
X_wl_scaled = scaler_final.transform(X_wl_full)

# Score
wl['p_liked'] = rf_final.predict_proba(X_wl_scaled)[:, 1]

print(f'Scored {len(wl)} watchlist films.')
print(f'Score distribution: min={wl["p_liked"].min():.3f}, '
      f'mean={wl["p_liked"].mean():.3f}, max={wl["p_liked"].max():.3f}')

In [ ]:
# ── Ranked watchlist table ────────────────────────────────────────────────────

display_cols = ['letterboxd_title', 'letterboxd_year', 'director',
                'genres', 'p_liked']

ranked = (
    wl[display_cols]
    .sort_values('p_liked', ascending=False)
    .reset_index(drop=True)
)
ranked.index += 1  # 1-indexed rank
ranked.columns = ['Title', 'Year', 'Director', 'Genres', 'P(Liked)']
ranked['P(Liked)'] = ranked['P(Liked)'].round(3)

print('=== TOP 25 WATCHLIST FILMS ===')
print(ranked.head(25).to_string())
print()
print('=== BOTTOM 10 (least likely to enjoy) ===')
print(ranked.tail(10).to_string())

In [ ]:
# ── Figure: top 30 watchlist films bar chart ──────────────────────────────────

top30 = ranked.head(30)

fig, ax = plt.subplots(figsize=(9, 10))
bars = ax.barh(
    top30['Title'][::-1],
    top30['P(Liked)'][::-1],
    color=plt.cm.RdYlGn(top30['P(Liked)'][::-1]),
    edgecolor='none'
)
ax.axvline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='p = 0.50')
ax.set_xlabel('Predicted Probability of Liking')
ax.set_title('Top 30 Watchlist Films by Predicted Preference', fontweight='bold')
ax.set_xlim(0, 1)
for bar, val in zip(bars[::-1], top30['P(Liked)']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=7.5)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('fig_watchlist_ranked.png', bbox_inches='tight')
plt.show()

# Save full ranked watchlist to CSV
ranked.to_csv('ranked_watchlist.csv')